# Smart Classroom Energy System — Revised Decision Tree Training
## Secondary dataset-based evaluation

This notebook trains and evaluates a Decision Tree classifier using the supplied `smart_lighting_dataset_2024.csv` file rather than the earlier synthetic dataset.

**Important label note:** the source CSV has a numeric target `lighting_action_class` with codes 0, 1, and 2. This notebook retains these codes as supplied. Before assigning human-readable labels such as ON/OFF/DIM, verify the definitions on the original Kaggle dataset page and cite that page in the project report.

**Reproducibility:** stratified 80/20 train/test split, random state 42. Preprocessing is encapsulated in a scikit-learn pipeline to prevent test-set leakage.


In [ ]:
# Install dependencies if needed in Kaggle/Jupyter
# %pip install pandas numpy scikit-learn matplotlib joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, precision_recall_fscore_support
)

RANDOM_STATE = 42


## 1. Load the required secondary dataset

In [ ]:
# In Kaggle, adjust this path to the uploaded dataset path if necessary.
DATA_PATH = Path("data/smart_lighting_dataset_2024.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("smart_lighting_dataset_2024.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/kaggle/input/smart-lighting-dataset-2024/smart_lighting_dataset_2024.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset CSV not found. Upload smart_lighting_dataset_2024.csv "
        "or update DATA_PATH to its exact location."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())


## 2. Inspect quality, class distribution, and target

In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))
print("Duplicate rows:", int(df.duplicated().sum()))
print("\nTarget distribution:")
display(df["lighting_action_class"].value_counts().sort_index().to_frame("count"))
print("\nNumeric summary:")
display(df.describe())


## 3. Prepare features and target

`hour_of_day` is extracted from the dataset timestamp. The selected predictors are environmental, occupancy, and temporal context fields. The target `lighting_action_class` is excluded from predictors.

The CSV contains three numeric target codes. They are retained as codes here; consult the original dataset description before translating them into named lighting actions.


In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["hour_of_day"] = df["timestamp"].dt.hour

FEATURES = [
    "ambient_light_lux", "motion_detected", "temperature_celsius",
    "occupancy_count", "hour_of_day", "day_of_week", "time_of_day",
    "weather_condition", "special_event_flag", "traffic_density",
    "avg_pedestrian_speed"
]
TARGET = "lighting_action_class"
CLASS_LABELS = [0, 1, 2]

required = set(FEATURES + [TARGET])
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df = df.dropna(subset=FEATURES + [TARGET]).copy()
X = df[FEATURES]
y = df[TARGET].astype(int)

print("Records after dropping rows with missing selected values:", len(df))
print("Feature count:", len(FEATURES))


## 4. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("\nTraining class proportions:")
display(y_train.value_counts(normalize=True).sort_index().to_frame("proportion"))
print("\nTesting class proportions:")
display(y_test.value_counts(normalize=True).sort_index().to_frame("proportion"))


## 5. Train the Decision Tree

In [ ]:
NUMERIC_FEATURES = [
    "ambient_light_lux", "motion_detected", "temperature_celsius",
    "occupancy_count", "hour_of_day", "special_event_flag",
    "traffic_density", "avg_pedestrian_speed"
]
CATEGORICAL_FEATURES = ["day_of_week", "time_of_day", "weather_condition"]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ("numeric", "passthrough", NUMERIC_FEATURES)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        criterion="gini",
        max_depth=5,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)
print("Decision Tree training completed.")


## 6. Evaluate on the held-out test set

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print("\nPrecision, recall, and F1-score for each class:")
print(classification_report(y_test, y_pred, labels=CLASS_LABELS, zero_division=0))


## 7. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=CLASS_LABELS)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Class 0", "Class 1", "Class 2"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

print("Confusion matrix (rows = actual, columns = predicted; order 0, 1, 2):")
print(cm)


## 8. Decision Tree visualization

In [ ]:
tree_model = model.named_steps["classifier"]
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

plt.figure(figsize=(22, 10))
plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=[str(c) for c in tree_model.classes_],
    filled=True,
    rounded=True,
    fontsize=7
)
plt.title("Decision Tree — Kaggle Secondary Dataset")
plt.tight_layout()
plt.show()


## 9. Save the model artifact

In [ ]:
OUTPUT_DIR = Path("models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

artifact = {
    "model": model,
    "features": FEATURES,
    "target": TARGET,
    "classes": list(model.classes_),
    "dataset_note": (
        "Trained on smart_lighting_dataset_2024.csv supplied by the project owner. "
        "Verify and cite the original Kaggle dataset page; numeric class names are retained."
    ),
    "random_state": RANDOM_STATE
}
MODEL_PATH = OUTPUT_DIR / "smart_classroom_decision_tree_kaggle.joblib"
joblib.dump(artifact, MODEL_PATH)
print("Saved model:", MODEL_PATH.resolve())


## 10. Dataset citation and label documentation

**Dataset source:** Kaggle secondary dataset, file supplied as `smart_lighting_dataset_2024.csv`.

**Citation status:** The exact Kaggle dataset page URL and creator details must be entered here after verifying the original page. Do not cite an unrelated Kaggle dataset. Preserve the original class codes until their definitions are verified.

**Label creation:** This training notebook does not create new labels or derive ON/OFF labels. It uses the existing `lighting_action_class` column as the target. The dataset's class definitions should be cited from its official dataset description.

## Interpretation and limitations
- Results describe performance on this dataset's stratified held-out split, not verified performance in a physical classroom.
- A high score does not by itself establish real-world generalization.
- The application must not claim physical relay control unless the controller and relay are connected and tested.
